## FRED rental-yield calibration

In [ ]:
import pandas as pd
import requests
from datetime import datetime

FRED_API_KEY = "YOUR API KEY"

def get_fred_data(series_id, api_key):
    """Fetch data from FRED API"""
    url = f"https://api.stlouisfed.org/fred/series/observations"
    params = {
        'series_id': series_id,
        'api_key': api_key,
        'file_type': 'json',
        'sort_order': 'desc',
        'limit': 10
    }

    response = requests.get(url, params=params)
    data = response.json()

    if 'observations' in data:
        df = pd.DataFrame(data['observations'])
        df['value'] = pd.to_numeric(df['value'], errors='coerce')
        df['date'] = pd.to_datetime(df['date'])
        return df
    else:
        print(f"Error fetching {series_id}: {data}")
        return None

def get_latest_annual_value(df, year=2024):
    """Get the most recent value for a given year"""
    if df is None:
        return None
    df_year = df[df['date'].dt.year == year]
    if len(df_year) > 0:
        return df_year.iloc[0]['value']
    else:
        return df.iloc[0]['value']

print("Fetching PCE data from FRED...\n")

# CORRECTED: Use NOMINAL National Housing PCE
# Try the nominal series first
national_housing_nominal = get_fred_data('DHUTRC1A027NBEA', FRED_API_KEY)

# If that fails, try alternative series
if national_housing_nominal is None:
    print("DHUTRC1A027NBEA not found, trying DHUTRG3A086NBEA...")
    national_housing_nominal = get_fred_data('DHUTRG3A086NBEA', FRED_API_KEY)

# Fetch other series
national_total_pce = get_fred_data('PCE', FRED_API_KEY)
florida_total_pce_data = get_fred_data('FLPCE', FRED_API_KEY)

# Extract 2024 values
if national_housing_nominal is not None:
    national_housing_2024 = get_latest_annual_value(national_housing_nominal, 2024)
else:
    # Fallback: Use the real series and note this in output
    national_housing_real = get_fred_data('DHUFRC1A027NBEA', FRED_API_KEY)
    national_housing_2024 = get_latest_annual_value(national_housing_real, 2024)

national_total_2024 = get_latest_annual_value(national_total_pce, 2024)
florida_total_millions = get_latest_annual_value(florida_total_pce_data, 2024)

# Convert Florida from millions to billions
florida_total_2024 = florida_total_millions / 1000

print(f"\n{'='*70}")
print("2024 PCE VALUES")
print(f"{'='*70}")
print(f"National Housing PCE:             ${national_housing_2024:,.1f}B")
print(f"National Total PCE:               ${national_total_2024:,.1f}B")
print(f"Florida Total PCE:                ${florida_total_2024:,.1f}B")

# Calculate national housing share
national_housing_share = national_housing_2024 / national_total_2024
print(f"\nNational Housing Share:           {national_housing_share*100:.2f}%")

# Apply to Florida
florida_housing_pce = florida_total_2024 * national_housing_share
print(f"Florida Housing PCE (implied):    ${florida_housing_pce:,.1f}B")

florida_housing_stock = 3500  # Billions

# Calculate rental yield
delta = florida_housing_pce / florida_housing_stock

print(f"\n{'='*70}")
print("RENTAL YIELD CALCULATION")
print(f"{'='*70}")
print(f"Florida Housing Stock Value:      ${florida_housing_stock:,.1f}B")
print(f"  Source: 10.0M units × $350K median")
print(f"  (Census ACS + Zillow HVI)")
print(f"Florida Housing PCE:              ${florida_housing_pce:,.1f}B")
print(f"\nRental Yield (δ):                 {delta:.4f}")
print(f"Rental Yield (percentage):        {delta*100:.2f}%")
print(f"{'='*70}")